In [ ]:
import torch
import math
import time
import functools
import contextlib
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

# 確認 GPU 環境


In [ ]:
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("警告：未檢測到 GPU。FlashAttention 需要 GPU 才能展現加速效果。")

GPU Device: NVIDIA A100-SXM4-80GB
CUDA Version: 12.8


## 實作注意力機制變體


In [ ]:
def run_naive_attention(q, k, v, is_causal=True):
    with sdpa_kernel(SDPBackend.MATH):
        return F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)

def run_flash_attention(q, k, v, is_causal=True):
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        return F.scaled_dot_product_attention(q, k, v, is_causal=is_causal)

## 數值正確性驗證


In [ ]:
# 1. 準備測試數據 (Shape: B, H, L, D)
# 這段程式碼主要用於生成模擬的 Attention：
# B (Batch Size): 批次大小。
# H (Num Heads): 注意力頭的數量。
# L (Sequence Length): 序列長度，即文本中 token 的數量。
# D (Head Dimension): 每個注意力頭的特徵維度。
B, H, L, D = 4, 8, 256, 64

#q, k, v: 分別代表 Query、Key 和 Value，這是計算 Attention 所需的三個基本元素。
q = torch.randn(B, H, L, D)
k = torch.randn(B, H, L, D)
v = torch.randn(B, H, L, D)

# 2. 執行不同的attention
out_naive = run_naive_attention(q, k, v)
out_flash = run_flash_attention(q, k, v)
#print(out_naive.shape)

# 3. 計算差異
diff = (out_naive - out_flash).abs().max().item()
print("最大差異:",diff)

最大差異: 7.748603820800781e-07


## 定義基準測試函式

對某種 attention 在 GPU 上重複執行多次，量平均執行時間。


In [ ]:
def benchmark_attention(mode, q, k, v, warmup=1, repeats=3):
    """
    測試指定 Attention 模式的執行時間。
    mode 可以是 'naive' 或 'flash'。
    Returns: avg_latency_ms
    """

    # 1. Warmup: Run the function 10 times without timing to stabilize the GPU state.
    for _ in range(warmup):
        if mode == 'naive':
            run_naive_attention(q, k, v)
        elif mode == 'flash':
            run_flash_attention(q, k, v)

    torch.cuda.synchronize() #如果不加同步，你後面做計時或記憶體統計時，可能會不準。

    # 2. Timing
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record() #從這裡開始算時間
    for _ in range(repeats):
        if mode == 'naive':
            run_naive_attention(q, k, v)
        elif mode == 'flash':
            run_flash_attention(q, k, v)
    end_event.record() #到這裡結束計時

    torch.cuda.synchronize() #等整個 repeats 次 attention 都真的完成，再讀時間。

    # 計算平均延遲 (毫秒)
    avg_latency_ms = start_event.elapsed_time(end_event) / repeats

    return avg_latency_ms

## 系統化實驗

In [ ]:
SEQ_LENS = [64, 128, 256, 512, 1024, 2048, 4096]
B, H, D = 4, 8, 64

print(f"Starting Benchmarks ... Params: B={B}, H={H}, D={D}")
print("-" * 90)
print(f"{'Seq Len':<10} | {'Naive (ms)':<12} | {'Flash (ms)':<12} | {'Flash Speedup':<15}")
print("-" * 90)

for L in SEQ_LENS:
    # Generate random input tensors
    q = torch.randn(B, H, L, D)
    k = torch.randn(B, H, L, D)
    v = torch.randn(B, H, L, D)

    # Naive Attention
    lat_naive = benchmark_attention('naive', q, k, v)
    math_str = f"{lat_naive:.2f}"

    torch.cuda.empty_cache()

    # Flash Attention
    lat_flash = benchmark_attention('flash', q, k, v)
    flash_str = f"{lat_flash:.2f}"

    torch.cuda.empty_cache()

    # Calculate Speedup (Naive / Flash)
    speedup = "-"
    s = lat_naive / lat_flash
    speedup = f"{s:.2f}x"

    # Print progress
    print(f"{L:<10} | {math_str:<12} | {flash_str:<12} | {speedup:<15}")

print("-" * 90)
print("Benchmark completed.")

Starting Benchmarks ... Params: B=4, H=8, D=64
------------------------------------------------------------------------------------------
Seq Len    | Naive (ms)   | Flash (ms)   | Flash Speedup  
------------------------------------------------------------------------------------------
64         | 1.11         | 0.21         | 5.19x          
128        | 1.35         | 0.50         | 2.69x          
256        | 6.77         | 1.41         | 4.81x          
512        | 24.42        | 4.91         | 4.97x          
1024       | 94.06        | 14.30        | 6.58x          
2048       | 369.78       | 45.64        | 8.10x          
4096       | 1605.56      | 169.17       | 9.49x          
------------------------------------------------------------------------------------------
Benchmark completed.


## 用真正的語言模型來測試 Flash Attention

In [ ]:
from transformers import pipeline

model_id = "google/gemma-3-4b-it"
print(f"使用真實模型: {model_id} ...")

print("下載模型中...")
pipe_eager = pipeline(
  "text-generation",
  model=model_id,
  model_kwargs={"attn_implementation": "eager"}
)

pipe_sdpa = pipeline(
  "text-generation",
  model=model_id,
  model_kwargs={"attn_implementation": "sdpa"}
)

使用真實模型: google/gemma-3-4b-it ...
下載模型中...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

In [ ]:
long_text = "FlashAttention is a fast and memory-efficient exact attention algorithm. " * 10000 #改大一點試試看
print(f"輸入長度: {len(long_text)} 字元\n")

輸入長度: 730000 字元



In [ ]:
print(f"=== 測試 Attention Implementation: Naive Attention ===")

print("正在進行 Warmup...")
pipe_eager("Hello", max_new_tokens=1)

print("開始測試...")
start_time = time.time()

pipe_eager(long_text,max_new_tokens=1)

torch.cuda.synchronize()

end_time = time.time()
latency = end_time - start_time
print(f"耗時: {latency:.4f} 秒")

# 清理記憶體
torch.cuda.empty_cache()

Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 測試 Attention Implementation: Naive Attention ===
正在進行 Warmup...
開始測試...


Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


OutOfMemoryError: CUDA out of memory. Tried to allocate 15.74 GiB. GPU 0 has a total capacity of 79.25 GiB of which 14.79 GiB is free. Including non-PyTorch memory, this process has 64.45 GiB memory in use. Of the allocated memory 63.90 GiB is allocated by PyTorch, and 55.78 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
print(f"=== 測試 Attention Implementation: Flash Attention ===")

print("正在進行 Warmup...")
pipe_sdpa("Hello", max_new_tokens=1)

print("開始測試...")
start_time = time.time()

pipe_sdpa(long_text,max_new_tokens=1)

torch.cuda.synchronize()

end_time = time.time()
latency = end_time - start_time
print(f"耗時: {latency:.4f} 秒")

# 清理記憶體
torch.cuda.empty_cache()

Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 測試 Attention Implementation: Flash Attention ===
正在進行 Warmup...
開始測試...
耗時: 1.2688 秒
